# Notebook 3 - Estrategia Momentum y Senales

Este notebook implementa la senal de momentum tipo MSCI Momentum con rebalanceo mensual en el ultimo dia habil (ultimo dato disponible del mes).

Objetivo:
- Construir senales de **R_12** y **R_6** con retornos logaritmicos y lag de 1 mes.
- Estandarizar cross-sectional por mes (Z_12 y Z_6).
- Construir score final y seleccionar TOP 20 activos por fecha de rebalanceo.
- Exportar el resultado a CSV para Notebook 4.


## Reglas del modelo (sin look-ahead)

Para cada fecha de rebalanceo mensual `t`:
- `R_12[t] = log(Pm[t-1] / Pm[t-13])`
- `R_6[t] = log(Pm[t-1] / Pm[t-7])`

Esto excluye el mes actual (`shift(1)`) y evita sesgo de look-ahead.

Normalizacion mensual cross-sectional:
- `Z_12[t, i] = (R_12[t, i] - mu_t) / sigma_t`
- `Z_6[t, i] = (R_6[t, i] - mu_t) / sigma_t`

Decision para `sigma_t = 0`:
- Se asigna `Z = 0` a los activos no nulos de esa fila.


In [37]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as fs

sns.set(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 200)

# Configuracion temporal de estrategia/senal
BACKTEST_START = pd.Timestamp("2015-01-01")
WARMUP_MONTHS = 13
WARMUP_START = BACKTEST_START - pd.DateOffset(months=WARMUP_MONTHS)
TOP_N = 20

# Filtros anti-outliers para limpiar seleccion final
MIN_DAILY_OBS_PRE = 120
MIN_LAST_PRICE_PRE = 5.0
MAX_P99_ABS_RET_PRE = 0.35
MAX_MAX_ABS_RET_PRE = 2.00
MAX_ANNUAL_VOL_PRE = 2.00


# Filtros anti-gaps (sin look-ahead)
MIN_PRICE_FILTER_MONTHLY_LAG = 5.0
USE_ADV20_FILTER = False
ADV20_WINDOW_DAYS = 20
ADV20_MIN_PERIODS = 10
MIN_ADV20_DOLLAR = 2_000_000.0
USE_JUMP63_FILTER = True
JUMP63_WINDOW_DAYS = 63
JUMP63_MIN_PERIODS = 20
MAX_JUMP63_ABS_RET = 0.70


USE_OPEN_CLOSE_ANOMALY_FILTER = True
OPEN_CLOSE_RATIO_MAX = 5.0
OPEN_CLOSE_ANOMALY_MAX_DAYS = 10
OPEN_CLOSE_MIN_OBS = 60
REQUIRE_OPEN_FOR_OC_FILTER = False

print("BACKTEST_START:", BACKTEST_START.date())
print("WARMUP_START (13 meses antes):", WARMUP_START.date())
print(
    "Filtros outliers ->",
    f"obs>={MIN_DAILY_OBS_PRE}, precio>={MIN_LAST_PRICE_PRE},",
    f"p99_abs_ret<={MAX_P99_ABS_RET_PRE}, max_abs_ret<={MAX_MAX_ABS_RET_PRE}, vol_anual<={MAX_ANNUAL_VOL_PRE}"
)
print(
    "Filtros anti-gaps ->",
    f"price_lag>={MIN_PRICE_FILTER_MONTHLY_LAG}, ",
    f"use_adv20={USE_ADV20_FILTER}, min_adv20=${MIN_ADV20_DOLLAR:,.0f}, ",
    f"use_jump63={USE_JUMP63_FILTER}, max_jump63={MAX_JUMP63_ABS_RET}, ",
    f"use_oc_anomaly={USE_OPEN_CLOSE_ANOMALY_FILTER}, oc_ratio_max={OPEN_CLOSE_RATIO_MAX}, ",
    f"oc_max_days={OPEN_CLOSE_ANOMALY_MAX_DAYS}"
)

# Filtro de volatilidad realizada (RV63) sin look-ahead (opcional)
# Modo actual: desactivado para usar momentum puro.
USE_RV63_FILTER = False
RV63_WINDOW_DAYS = 63
RV63_MIN_PERIODS = 40
RV63_MAX_PERCENTILE = 0.90


BACKTEST_START: 2015-01-01
WARMUP_START (13 meses antes): 2013-12-01
Filtros outliers -> obs>=120, precio>=5.0, p99_abs_ret<=0.35, max_abs_ret<=2.0, vol_anual<=2.0
Filtros anti-gaps -> price_lag>=5.0,  use_adv20=False, min_adv20=$2,000,000,  use_jump63=True, max_jump63=0.7,  use_oc_anomaly=True, oc_ratio_max=5.0,  oc_max_days=10


## Funciones reutilizables (para Notebook 4)

Se definen funciones pequenas para reutilizar el pipeline de senales.


In [38]:
def to_wide_close(df):
    """
    Convierte input a matriz wide de precios close:
    index = fecha, columns = tickers.
    Acepta:
    - formato long con [date, ticker/symbol, close/adj_close]
    - formato wide con DatetimeIndex
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise ValueError("Input vacio o no es DataFrame.")

    x = df.copy()
    x.columns = [str(c).strip().lower() for c in x.columns]

    date_cands = ["date", "datetime", "timestamp", "fecha"]
    ticker_cands = ["ticker", "symbol", "asset", "activo"]
    close_cands = ["adj_close", "close", "unadjusted_close", "price"]

    date_col = next((c for c in date_cands if c in x.columns), None)
    ticker_col = next((c for c in ticker_cands if c in x.columns), None)
    close_col = next((c for c in close_cands if c in x.columns), None)

    # Caso long: date + ticker + close
    if date_col is not None and ticker_col is not None and close_col is not None:
        y = x[[date_col, ticker_col, close_col]].copy()
        y[date_col] = pd.to_datetime(y[date_col], errors="coerce")
        y[ticker_col] = y[ticker_col].astype(str).str.upper()
        y = y.dropna(subset=[date_col, ticker_col, close_col])
        px = y.pivot_table(index=date_col, columns=ticker_col, values=close_col, aggfunc="last")
        px = px.sort_index()
        return px

    # Caso wide con datetime index
    y = df.copy()
    if not isinstance(y.index, pd.DatetimeIndex):
        if date_col is None:
            raise ValueError("No encuentro columna fecha ni DatetimeIndex para construir matriz de precios.")
        y[date_col] = pd.to_datetime(y[date_col], errors="coerce")
        y = y.dropna(subset=[date_col]).set_index(date_col)

    for c in y.columns:
        y[c] = pd.to_numeric(y[c], errors="coerce")

    y = y.select_dtypes(include=[np.number]).sort_index()
    y.columns = [str(c).strip().upper() for c in y.columns]
    y = y.dropna(axis=1, how="all")
    return y


def to_wide_volume(df):
    """
    Extrae volumen diario en formato wide (date x ticker) desde formato long.
    Si no existe columna de volumen, devuelve None.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        return None

    x = df.copy()
    x.columns = [str(c).strip().lower() for c in x.columns]

    date_cands = ["date", "datetime", "timestamp", "fecha"]
    ticker_cands = ["ticker", "symbol", "asset", "activo"]
    vol_cands = ["volume", "vol", "volumen", "shares_volume", "n_volume"]

    date_col = next((c for c in date_cands if c in x.columns), None)
    ticker_col = next((c for c in ticker_cands if c in x.columns), None)
    vol_col = next((c for c in vol_cands if c in x.columns), None)

    if date_col is None or ticker_col is None or vol_col is None:
        return None

    y = x[[date_col, ticker_col, vol_col]].copy()
    y[date_col] = pd.to_datetime(y[date_col], errors="coerce")
    y[ticker_col] = y[ticker_col].astype(str).str.upper()
    y[vol_col] = pd.to_numeric(y[vol_col], errors="coerce")
    y = y.dropna(subset=[date_col, ticker_col, vol_col])

    if len(y) == 0:
        return None

    vw = y.pivot_table(index=date_col, columns=ticker_col, values=vol_col, aggfunc="last")
    vw = vw.sort_index().replace([np.inf, -np.inf], np.nan)
    vw = vw.dropna(axis=1, how="all")
    return vw


def to_wide_open(df):
    """
    Extrae precio OPEN diario en formato wide (date x ticker) desde formato long.
    Si no existe columna de open, devuelve None.
    """
    if not isinstance(df, pd.DataFrame) or df.empty:
        return None

    x = df.copy()
    x.columns = [str(c).strip().lower() for c in x.columns]

    date_cands = ["date", "datetime", "timestamp", "fecha"]
    ticker_cands = ["ticker", "symbol", "asset", "activo"]
    open_cands = ["open", "open_raw", "opening_price"]

    date_col = next((c for c in date_cands if c in x.columns), None)
    ticker_col = next((c for c in ticker_cands if c in x.columns), None)
    open_col = next((c for c in open_cands if c in x.columns), None)

    if date_col is None or ticker_col is None or open_col is None:
        return None

    y = x[[date_col, ticker_col, open_col]].copy()
    y[date_col] = pd.to_datetime(y[date_col], errors="coerce")
    y[ticker_col] = y[ticker_col].astype(str).str.upper()
    y[open_col] = pd.to_numeric(y[open_col], errors="coerce")
    y = y.dropna(subset=[date_col, ticker_col, open_col])

    if len(y) == 0:
        return None

    ow = y.pivot_table(index=date_col, columns=ticker_col, values=open_col, aggfunc="last")
    ow = ow.sort_index().replace([np.inf, -np.inf], np.nan)
    ow = ow.dropna(axis=1, how="all")
    return ow


def to_monthly_last(px_close):
    """
    Convierte precios a mensual usando ultimo dato disponible de cada mes.
    - Si parece diario/intradiario: resample BM + last.
    - Si ya parece mensual: colapsa por mes y toma ultimo disponible.
    """
    if not isinstance(px_close.index, pd.DatetimeIndex):
        raise TypeError("px_close debe tener DatetimeIndex.")

    px = px_close.sort_index().copy()
    px = px[~px.index.duplicated(keep="last")]
    px = px.dropna(axis=1, how="all")

    obs_per_month = px.groupby(px.index.to_period("M")).size()
    is_daily_like = obs_per_month.median() > 1

    if is_daily_like:
        Pm = px.resample("BM").last()
    else:
        Pm = px.groupby(px.index.to_period("M")).last()
        Pm.index = Pm.index.to_timestamp("M")

    Pm = Pm.sort_index().dropna(axis=1, how="all")
    return Pm


def compute_momentum(Pm):
    """
    Senales momentum con lag 1 mes (sin look-ahead):
    R12[t] = log(Pm[t-1]/Pm[t-13])
    R6[t]  = log(Pm[t-1]/Pm[t-7])
    """
    R12 = np.log(Pm.shift(1) / Pm.shift(13))
    R6 = np.log(Pm.shift(1) / Pm.shift(7))
    return R12, R6




def compute_realized_vol_63(px_daily, window=63, min_periods=40):
    # Volatilidad realizada diaria anualizada (rolling).
    # Para evitar look-ahead en rebalance mensual, usar luego shift(1).
    if not isinstance(px_daily, pd.DataFrame):
        raise TypeError("px_daily debe ser DataFrame para RV63.")
    if px_daily.empty:
        return pd.DataFrame(index=px_daily.index, columns=px_daily.columns, dtype=float)

    ret_d = px_daily.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    rv = ret_d.rolling(window=int(window), min_periods=int(min_periods)).std() * np.sqrt(252)
    return rv

def cross_sectional_zscore(df):
    """
    Z-score cross-sectional por fecha (fila), ignorando NaN.
    Regla std=0: Z=0 para valores no nulos de esa fila.
    """
    mu = df.mean(axis=1, skipna=True)
    sigma = df.std(axis=1, ddof=0, skipna=True)

    z = df.sub(mu, axis=0).div(sigma.replace(0, np.nan), axis=0)

    zero_std_idx = sigma[sigma == 0].index
    if len(zero_std_idx) > 0:
        z.loc[zero_std_idx] = np.where(df.loc[zero_std_idx].notna(), 0.0, np.nan)

    return z


def build_scores_and_select(R12, R6, top_n=20, hold_last_weights=True,
                            rv63=None, use_rv63_filter=True, rv63_max_percentile=0.90,
                            price_lag=None, min_price_lag=10.0,
                            adv20=None, use_adv20_filter=True, min_adv20=5_000_000.0,
                            jump63=None, use_jump63_filter=True, max_jump63_abs_ret=0.40,
                            oc_anomaly_count=None, oc_obs_count=None,
                            use_open_close_anomaly_filter=True,
                            open_close_anomaly_max_days=3, open_close_min_obs=120):
    """
    Construye score, rankea y selecciona top_n por fecha.
    - Sin GLD (se asume GLD ya fuera de columnas).
    - Empates: score desc, ticker asc (estable y reproducible).
    - Filtros opcionales sin look-ahead: RV63, precio minimo lagged, ADV20, salto maximo 63d y anomalias OPEN/CLOSE.
    - Si en una fecha hay <top_n candidatos validos:
        * hold_last_weights=True: mantiene pesos del mes anterior.
        * si no hay cartera previa: se queda en cash.
    """
    if R12.shape != R6.shape:
        raise ValueError("R12 y R6 deben tener misma forma.")

    z12 = cross_sectional_zscore(R12)
    z6 = cross_sectional_zscore(R6)
    score = 0.5 * (z12 + z6)

    tickers = score.columns
    idx = score.index

    if rv63 is not None:
        if not isinstance(rv63, pd.DataFrame):
            raise TypeError("rv63 debe ser DataFrame o None.")
        rv63 = rv63.reindex(index=idx, columns=tickers)

    if price_lag is not None:
        if not isinstance(price_lag, pd.DataFrame):
            raise TypeError("price_lag debe ser DataFrame o None.")
        price_lag = price_lag.reindex(index=idx, columns=tickers)

    if adv20 is not None:
        if not isinstance(adv20, pd.DataFrame):
            raise TypeError("adv20 debe ser DataFrame o None.")
        adv20 = adv20.reindex(index=idx, columns=tickers)

    if jump63 is not None:
        if not isinstance(jump63, pd.DataFrame):
            raise TypeError("jump63 debe ser DataFrame o None.")
        jump63 = jump63.reindex(index=idx, columns=tickers)

    if oc_anomaly_count is not None:
        if not isinstance(oc_anomaly_count, pd.DataFrame):
            raise TypeError("oc_anomaly_count debe ser DataFrame o None.")
        oc_anomaly_count = oc_anomaly_count.reindex(index=idx, columns=tickers)

    if oc_obs_count is not None:
        if not isinstance(oc_obs_count, pd.DataFrame):
            raise TypeError("oc_obs_count debe ser DataFrame o None.")
        oc_obs_count = oc_obs_count.reindex(index=idx, columns=tickers)

    valid_score_count = score.notna().sum(axis=1)
    first_rebalance = valid_score_count[valid_score_count >= top_n].index.min()
    if pd.isna(first_rebalance):
        raise ValueError(f"No hay ninguna fecha con al menos {top_n} scores validos.")

    tickers = score.columns.tolist()
    dates = score.index.tolist()

    selection_wide = pd.DataFrame(0.0, index=dates, columns=tickers)
    long_rows = []

    prev_weights = pd.Series(0.0, index=tickers)
    prev_ranked = []
    invested = False

    for dt in dates:
        row = pd.DataFrame({
            "ticker": tickers,
            "score": score.loc[dt].values,
            "z12": z12.loc[dt].values,
            "z6": z6.loc[dt].values,
            "r12": R12.loc[dt].values,
            "r6": R6.loc[dt].values,
            "rv63": rv63.loc[dt].values if rv63 is not None else np.nan,
            "price_lag": price_lag.loc[dt].values if price_lag is not None else np.nan,
            "adv20": adv20.loc[dt].values if adv20 is not None else np.nan,
            "jump63": jump63.loc[dt].values if jump63 is not None else np.nan,
            "oc_anomaly_count": oc_anomaly_count.loc[dt].values if oc_anomaly_count is not None else np.nan,
            "oc_obs_count": oc_obs_count.loc[dt].values if oc_obs_count is not None else np.nan,
        }).dropna(subset=["score"])

        rv63_cutoff = np.nan
        row_candidates = row.copy()

        if use_rv63_filter and (rv63 is not None):
            row_with_rv = row_candidates.dropna(subset=["rv63"]).copy()
            if len(row_with_rv) > 0:
                rv63_cutoff = float(row_with_rv["rv63"].quantile(rv63_max_percentile))
                row_candidates = row_with_rv[row_with_rv["rv63"] <= rv63_cutoff].copy()
            else:
                row_candidates = row_with_rv

        if price_lag is not None and min_price_lag is not None:
            row_candidates = row_candidates.dropna(subset=["price_lag"])
            row_candidates = row_candidates[row_candidates["price_lag"] >= float(min_price_lag)].copy()

        if use_adv20_filter and (adv20 is not None) and (min_adv20 is not None):
            row_candidates = row_candidates.dropna(subset=["adv20"])
            row_candidates = row_candidates[row_candidates["adv20"] >= float(min_adv20)].copy()

        if use_jump63_filter and (jump63 is not None) and (max_jump63_abs_ret is not None):
            row_candidates = row_candidates.dropna(subset=["jump63"])
            row_candidates = row_candidates[row_candidates["jump63"] <= float(max_jump63_abs_ret)].copy()

        if use_open_close_anomaly_filter and (oc_anomaly_count is not None) and (oc_obs_count is not None):
            row_candidates = row_candidates.dropna(subset=["oc_anomaly_count", "oc_obs_count"])
            row_candidates = row_candidates[row_candidates["oc_obs_count"] >= float(open_close_min_obs)].copy()
            row_candidates = row_candidates[row_candidates["oc_anomaly_count"] <= float(open_close_anomaly_max_days)].copy()

        has_enough = len(row_candidates) >= top_n

        if has_enough:
            ranked = row_candidates.sort_values(["score", "ticker"], ascending=[False, True]).head(top_n).reset_index(drop=True)
            ranked["rank"] = np.arange(1, top_n + 1)

            current_weights = pd.Series(0.0, index=tickers)
            current_weights.loc[ranked["ticker"].tolist()] = 1.0 / top_n

            prev_weights = current_weights.copy()
            prev_ranked = ranked["ticker"].tolist()
            invested = True
            rebalanced_flag = 1
        else:
            if hold_last_weights and invested:
                current_weights = prev_weights.copy()
                ranked = pd.DataFrame({
                    "ticker": prev_ranked,
                    "rank": np.arange(1, len(prev_ranked) + 1),
                })

                map_score = score.loc[dt].to_dict()
                map_z12 = z12.loc[dt].to_dict()
                map_z6 = z6.loc[dt].to_dict()
                map_r12 = R12.loc[dt].to_dict()
                map_r6 = R6.loc[dt].to_dict()
                map_rv63 = rv63.loc[dt].to_dict() if rv63 is not None else {}
                map_price = price_lag.loc[dt].to_dict() if price_lag is not None else {}
                map_adv20 = adv20.loc[dt].to_dict() if adv20 is not None else {}
                map_jump63 = jump63.loc[dt].to_dict() if jump63 is not None else {}
                map_oc_anom = oc_anomaly_count.loc[dt].to_dict() if oc_anomaly_count is not None else {}
                map_oc_obs = oc_obs_count.loc[dt].to_dict() if oc_obs_count is not None else {}

                ranked["score"] = ranked["ticker"].map(map_score)
                ranked["z12"] = ranked["ticker"].map(map_z12)
                ranked["z6"] = ranked["ticker"].map(map_z6)
                ranked["r12"] = ranked["ticker"].map(map_r12)
                ranked["r6"] = ranked["ticker"].map(map_r6)
                ranked["rv63"] = ranked["ticker"].map(map_rv63)
                ranked["price_lag"] = ranked["ticker"].map(map_price)
                ranked["adv20"] = ranked["ticker"].map(map_adv20)
                ranked["jump63"] = ranked["ticker"].map(map_jump63)
                ranked["oc_anomaly_count"] = ranked["ticker"].map(map_oc_anom)
                ranked["oc_obs_count"] = ranked["ticker"].map(map_oc_obs)
                rebalanced_flag = 0
            else:
                current_weights = pd.Series(0.0, index=tickers)
                ranked = pd.DataFrame(columns=[
                    "ticker", "rank", "score", "z12", "z6", "r12", "r6", "rv63", "price_lag", "adv20", "jump63",
                    "oc_anomaly_count", "oc_obs_count"
                ])
                rebalanced_flag = 0

        if len(ranked) > 0:
            ranked["rv63_cutoff"] = rv63_cutoff
            ranked["rv63_filter_pass"] = (
                (ranked["rv63"] <= rv63_cutoff).astype(int) if (use_rv63_filter and pd.notna(rv63_cutoff)) else np.nan
            )
            ranked["price_filter_pass"] = (
                (ranked["price_lag"] >= float(min_price_lag)).astype(int)
                if (price_lag is not None and min_price_lag is not None)
                else np.nan
            )
            ranked["adv20_filter_pass"] = (
                (ranked["adv20"] >= float(min_adv20)).astype(int)
                if (use_adv20_filter and adv20 is not None and min_adv20 is not None)
                else np.nan
            )
            ranked["jump63_filter_pass"] = (
                (ranked["jump63"] <= float(max_jump63_abs_ret)).astype(int)
                if (use_jump63_filter and jump63 is not None and max_jump63_abs_ret is not None)
                else np.nan
            )
            ranked["oc_anomaly_filter_pass"] = (
                ((ranked["oc_obs_count"] >= float(open_close_min_obs)) & (ranked["oc_anomaly_count"] <= float(open_close_anomaly_max_days))).astype(int)
                if (use_open_close_anomaly_filter and oc_anomaly_count is not None and oc_obs_count is not None)
                else np.nan
            )

        selection_wide.loc[dt] = current_weights.values

        if invested and len(ranked) > 0:
            if "score" not in ranked.columns:
                ranked = ranked.merge(row, on="ticker", how="left")

            for _, r in ranked.iterrows():
                tkr = r["ticker"]
                long_rows.append({
                    "rebalance_date": dt,
                    "ticker": tkr,
                    "rank": int(r["rank"]),
                    "score": r.get("score", np.nan),
                    "z12": r.get("z12", np.nan),
                    "z6": r.get("z6", np.nan),
                    "r12": r.get("r12", np.nan),
                    "r6": r.get("r6", np.nan),
                    "rv63": r.get("rv63", np.nan),
                    "rv63_cutoff": r.get("rv63_cutoff", np.nan),
                    "rv63_filter_pass": r.get("rv63_filter_pass", np.nan),
                    "price_lag": r.get("price_lag", np.nan),
                    "price_filter_pass": r.get("price_filter_pass", np.nan),
                    "adv20": r.get("adv20", np.nan),
                    "adv20_filter_pass": r.get("adv20_filter_pass", np.nan),
                    "jump63": r.get("jump63", np.nan),
                    "jump63_filter_pass": r.get("jump63_filter_pass", np.nan),
                    "oc_anomaly_count": r.get("oc_anomaly_count", np.nan),
                    "oc_obs_count": r.get("oc_obs_count", np.nan),
                    "oc_anomaly_filter_pass": r.get("oc_anomaly_filter_pass", np.nan),
                    "weight": float(current_weights.get(tkr, 0.0)),
                    "rebalanced": rebalanced_flag,
                })

    selection_long = pd.DataFrame(long_rows).sort_values(["rebalance_date", "rank", "ticker"]).reset_index(drop=True)
    return selection_wide, selection_long



## Carga de datos desde NB1/NB2 (sin redescarga)

Se intenta leer primero variables en memoria (NB2) y luego ficheros de `data/`, `output/` y `outputs/` (parquet/csv).
No se anade GLD; el universo se trabaja sin GLD.


In [39]:
def _is_valid_file(path, min_size=100):
    local_fs = fs.LocalFileSystem()
    try:
        info = local_fs.get_file_info(path)
        if info.type != fs.FileType.File:
            return False
        if info.size is None:
            return False
        return info.size > min_size
    except Exception:
        return False


def _scan_candidate_files():
    local_fs = fs.LocalFileSystem()

    roots = [
        "data", "output", "outputs", ".",
        "..", "../data", "../output", "../outputs",
        r"C:\\Users\\alons\\Desktop",
    ]

    found = []
    seen = set()

    # Rutas directas tipicas del proyecto/curso
    fixed_candidates = [
        r"C:\\Users\\alons\\Desktop\\Práctica 7\\sp500_history.parquet",
        r"C:\\Users\\alons\\Desktop\\Practica 7\\sp500_history.parquet",
        r"data\\raw\\sp500_history.parquet",
        r"..\\data\\raw\\sp500_history.parquet",
    ]

    for path in fixed_candidates:
        if path not in seen and _is_valid_file(path):
            info = local_fs.get_file_info(path)
            found.append((path, info.size))
            seen.add(path)

    for root in roots:
        try:
            infos = local_fs.get_file_info(fs.FileSelector(root, recursive=True))
        except Exception:
            continue

        for info in infos:
            if info.type != fs.FileType.File:
                continue

            path = info.path
            path_l = path.lower()

            if not (path_l.endswith(".parquet") or path_l.endswith(".csv")):
                continue

            if "notebook_3_estrategia_momentum_senales" in path_l:
                continue
            if "selected_top20_by_rebalance" in path_l:
                continue

            if info.size is None or info.size <= 100:
                continue

            if path not in seen:
                found.append((path, info.size))
                seen.add(path)

    def _score(path, size):
        p = path.lower()
        s = 0
        if "processed" in p or "procesado" in p or "prepared" in p or "prepar" in p:
            s += 8
        if "sp500" in p or "history" in p:
            s += 10
        if "data" in p:
            s += 2
        if p.endswith(".parquet"):
            s += 3
        if size is not None and size > 1024:
            s += 2
        return s

    found = sorted(found, key=lambda x: (_score(x[0], x[1]), x[1]), reverse=True)
    return [x[0] for x in found]


def _read_any_table(path):
    path_l = path.lower()
    try:
        if path_l.endswith(".parquet"):
            return pq.read_table(path).to_pandas()
        if path_l.endswith(".csv"):
            return pd.read_csv(path)
    except Exception:
        return None
    return None


def _to_close_wide(df):
    if df is None or len(df) == 0:
        return None

    work = df.copy()
    work.columns = [str(c).strip().lower() for c in work.columns]

    date_cands = ["date", "datetime", "timestamp", "fecha"]
    ticker_cands = ["symbol", "ticker", "asset", "activo"]
    close_cands = ["adj_close", "close", "unadjusted_close", "price"]

    date_col = next((c for c in date_cands if c in work.columns), None)
    ticker_col = next((c for c in ticker_cands if c in work.columns), None)
    close_col = next((c for c in close_cands if c in work.columns), None)

    wide = None

    if date_col is not None and ticker_col is not None and close_col is not None:
        tmp = work[[date_col, ticker_col, close_col]].copy()
        tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
        tmp[ticker_col] = tmp[ticker_col].astype(str).str.upper()
        tmp = tmp.dropna(subset=[date_col, ticker_col, close_col])
        wide = tmp.pivot_table(index=date_col, columns=ticker_col, values=close_col, aggfunc="last")

    elif date_col is not None:
        tmp = work.copy()
        tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
        tmp = tmp.dropna(subset=[date_col]).set_index(date_col)

        for c in tmp.columns:
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

        num_cols = tmp.select_dtypes(include=[np.number]).columns.tolist()
        if len(num_cols) >= 2:
            wide = tmp[num_cols].copy()
            wide.columns = [str(c).strip().upper() for c in wide.columns]

    elif isinstance(df.index, pd.DatetimeIndex):
        tmp = df.copy()
        for c in tmp.columns:
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
        num_cols = tmp.select_dtypes(include=[np.number]).columns.tolist()
        if len(num_cols) >= 2:
            wide = tmp[num_cols].copy()
            wide.columns = [str(c).strip().upper() for c in wide.columns]

    if wide is None or wide.shape[1] == 0:
        return None

    wide = wide.replace([np.inf, -np.inf], np.nan)
    wide.index = pd.to_datetime(wide.index, errors="coerce")
    wide = wide[~wide.index.isna()]
    wide = wide.sort_index()
    wide = wide[~wide.index.duplicated(keep="last")]
    wide = wide.dropna(axis=1, how="all")

    if wide.shape[1] == 0:
        return None

    return wide


def _load_from_nb2_globals():
    candidate_names = [
        "datos_analisis_df",
        "df_prepared",
        "datos_base_df",
        "datos_auditoria_df",
        "df",
        "data",
    ]

    for name in candidate_names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            px = _to_close_wide(globals()[name])
            if px is not None and px.shape[1] > 0:
                return px, f"Notebook2::{name}"

    return None, None


def load_prepared_close_prices(min_assets=20):
    # 1) Primero: datos en memoria de NB2
    px_mem, mem_source = _load_from_nb2_globals()
    if px_mem is not None:
        if px_mem.shape[1] >= min_assets and px_mem.shape[0] >= 260:
            print(f"Fuente seleccionada: {mem_source}")
            return px_mem, mem_source
        print(f"Fuente en memoria detectada ({mem_source}) pero con cobertura baja: {px_mem.shape}")

    # 2) PARQUET_PATH si viene de NB1
    if "PARQUET_PATH" in globals() and isinstance(globals()["PARQUET_PATH"], str):
        df_try = _read_any_table(globals()["PARQUET_PATH"])
        px_try = _to_close_wide(df_try)
        if px_try is not None and px_try.shape[1] > 0:
            print("Fuente seleccionada: PARQUET_PATH")
            return px_try, "PARQUET_PATH"

    # 3) Fallback por escaneo de ficheros
    candidates = _scan_candidate_files()

    if len(candidates) == 0:
        raise FileNotFoundError(
            "No se encontraron fuentes validas. Ejecuta NB2 en el mismo kernel o verifica que exista sp500_history.parquet."
        )

    best_px = None
    best_path = None
    best_score = -1

    for path in candidates:
        df = _read_any_table(path)
        px = _to_close_wide(df)

        if px is None:
            continue

        score = px.shape[0] * px.shape[1]

        if px.shape[1] >= min_assets and px.shape[0] >= 260:
            print(f"Fuente seleccionada: {path}")
            return px, path

        if score > best_score:
            best_score = score
            best_px = px
            best_path = path

    if best_px is not None:
        print(f"Fuente seleccionada (mejor disponible): {best_path}")
        return best_px, best_path

    raise ValueError(
        "No fue posible construir una matriz diaria de close. Ejecuta NB2 en el mismo kernel o verifica archivos de salida."
    )


px_close_daily, source_path = load_prepared_close_prices(min_assets=20)

print("Shape diario (close):", px_close_daily.shape)
print("Rango diario:", px_close_daily.index.min(), "->", px_close_daily.index.max())
print("Numero de activos diarios:", px_close_daily.shape[1])
print("Fuente usada:", source_path)


Fuente seleccionada: C:\\Users\\alons\\Desktop\\Práctica 7\\sp500_history.parquet
Shape diario (close): (9087, 1289)
Rango diario: 1990-01-02 00:00:00 -> 2026-01-30 00:00:00
Numero de activos diarios: 1289
Fuente usada: C:\\Users\\alons\\Desktop\\Práctica 7\\sp500_history.parquet


## Universo final: activos vivos (sin GLD)

Regla:
- Se toma universo de activos con datos en los ultimos 13 meses del panel mensual.
- GLD queda explicitamente fuera del universo.
- Si el universo final tiene menos de 20 activos, se lanza error claro.


In [40]:
# Construccion de Pm mensual y universo limpio (sin GLD, sin sesgo futuro)

# 1) Intentamos tomar precios desde variables ya cargadas en memoria.
# Prioridad: datos diarios (para RV63) antes que Pm mensual.
if "px_close_daily" in globals() and isinstance(px_close_daily, pd.DataFrame):
    prices_input = px_close_daily.copy()
elif "datos_analisis_df" in globals() and isinstance(datos_analisis_df, pd.DataFrame):
    prices_input = datos_analisis_df.copy()
elif "datos_base_df" in globals() and isinstance(datos_base_df, pd.DataFrame):
    prices_input = datos_base_df.copy()
elif "df_prepared" in globals() and isinstance(df_prepared, pd.DataFrame):
    prices_input = df_prepared.copy()
elif "Pm" in globals() and isinstance(Pm, pd.DataFrame):
    prices_input = Pm.copy()
else:
    px_close_daily, source_path = load_prepared_close_prices(min_assets=20)
    prices_input = px_close_daily.copy()

# 2) Si viene en formato long y trae bandera sp500, filtramos ex-ante
if isinstance(prices_input, pd.DataFrame):
    col_map = {str(c).strip().lower(): c for c in prices_input.columns}
    date_col = next((col_map[k] for k in ["date", "datetime", "timestamp", "fecha"] if k in col_map), None)
    ticker_col = next((col_map[k] for k in ["ticker", "symbol", "asset", "activo"] if k in col_map), None)
    close_col = next((col_map[k] for k in ["adj_close", "close", "unadjusted_close", "price"] if k in col_map), None)
    flag_col = next((col_map[k] for k in ["in_sp500", "is_sp500", "sp500_flag"] if k in col_map), None)

    if date_col is not None and ticker_col is not None and close_col is not None and flag_col is not None:
        tmp = prices_input.copy()
        flag = tmp[flag_col]
        if str(flag.dtype).lower() == "bool":
            mask = flag.fillna(False)
        else:
            mask = pd.to_numeric(flag, errors="coerce").fillna(0) > 0

        tmp = tmp[mask].copy()
        if len(tmp) > 0:
            prices_input = tmp
            print("Filtro in_sp500 aplicado antes de construir matriz de precios.")

vol_daily_input = to_wide_volume(prices_input)
if vol_daily_input is not None and len(vol_daily_input) > 0:
    vol_daily_input = vol_daily_input.sort_index().replace([np.inf, -np.inf], np.nan)
open_daily_input = to_wide_open(prices_input)
if open_daily_input is not None and len(open_daily_input) > 0:
    open_daily_input = open_daily_input.sort_index().replace([np.inf, -np.inf], np.nan)
elif "source_path" in globals() and isinstance(source_path, str):
    try:
        raw_src = _read_any_table(source_path)
        open_daily_input = to_wide_open(raw_src)
        if open_daily_input is not None and len(open_daily_input) > 0:
            open_daily_input = open_daily_input.sort_index().replace([np.inf, -np.inf], np.nan)
            print("OPEN diario cargado desde source_path para filtro de anomalias OPEN/CLOSE.")
    except Exception:
        open_daily_input = None
px_close = to_wide_close(prices_input)
px_close = px_close.sort_index().replace([np.inf, -np.inf], np.nan)
Pm_full = to_monthly_last(px_close)

vol_daily = None
if "vol_daily_input" in locals() and vol_daily_input is not None and len(vol_daily_input) > 0:
    vol_daily = vol_daily_input.reindex(index=px_close.index, columns=px_close.columns)

open_daily = None
if "open_daily_input" in locals() and open_daily_input is not None and len(open_daily_input) > 0:
    open_daily = open_daily_input.reindex(index=px_close.index, columns=px_close.columns)

# 3) Warm-up: mantenemos solo desde 13 meses antes del inicio del backtest
Pm_full = Pm_full[Pm_full.index >= WARMUP_START].copy()
px_close = px_close[px_close.index >= WARMUP_START].copy()

if Pm_full.shape[0] < 14:
    raise ValueError("No hay suficientes meses de historia para calcular R12 con lag (se requieren al menos 14 meses).")

# 4) Limpieza de tickers: fuera GLD, fuera sufijo Q de 5 letras, formato ticker razonable
Pm_full = Pm_full.drop(columns=["GLD"], errors="ignore")

tickers = pd.Index([str(c).strip().upper() for c in Pm_full.columns])
valid_format = tickers.to_series(index=tickers).str.fullmatch(r"[A-Z]{1,5}(\.[A-Z])?").fillna(False)
ticker_len = tickers.to_series(index=tickers).str.len().fillna(0)
otc_q_like = tickers.to_series(index=tickers).str.endswith("Q") & (ticker_len >= 5)

clean_tickers = sorted(set(tickers[valid_format & (~otc_q_like)].tolist()))
Pm_full = Pm_full.reindex(columns=clean_tickers)
px_close = px_close.reindex(columns=clean_tickers)
if vol_daily is not None:
    vol_daily = vol_daily.reindex(columns=clean_tickers)
if open_daily is not None:
    open_daily = open_daily.reindex(columns=clean_tickers)

# 5) Intentamos intersecar con universo S&P desde raw parquet (si existe flag)
def _load_sp500_set_from_raw():
    candidate_paths = []
    if "PARQUET_PATH" in globals() and isinstance(PARQUET_PATH, str):
        candidate_paths.append(PARQUET_PATH)
    if "source_path" in globals() and isinstance(source_path, str):
        candidate_paths.append(source_path)

    known = [
        r"C:\Users\alons\Desktop\Pr?ctica 7\sp500_history.parquet",
        r"C:\Users\alons\Desktop\Practica 7\sp500_history.parquet",
        r"data\raw\sp500_history.parquet",
        r"..\data\raw\sp500_history.parquet",
    ]
    for k in known:
        if k not in candidate_paths:
            candidate_paths.append(k)

    local_fs = fs.LocalFileSystem()

    for path in candidate_paths:
        try:
            info = local_fs.get_file_info(path)
            if info.type != fs.FileType.File:
                continue

            table = pq.read_table(path)
            cols_lower = [str(c).strip().lower() for c in table.column_names]

            dcol = next((c for c in ["date", "datetime", "timestamp", "fecha"] if c in cols_lower), None)
            tcol = next((c for c in ["symbol", "ticker", "asset", "activo"] if c in cols_lower), None)
            fcol = next((c for c in ["in_sp500", "is_sp500", "sp500_flag"] if c in cols_lower), None)

            if dcol is None or tcol is None or fcol is None:
                continue

            use_cols = [dcol, tcol, fcol]
            t2 = pq.read_table(path, columns=use_cols)
            raw = t2.to_pandas()

            raw.columns = [str(c).strip().lower() for c in raw.columns]
            raw[dcol] = pd.to_datetime(raw[dcol], errors="coerce")
            raw = raw.dropna(subset=[dcol, tcol])
            raw = raw[(raw[dcol] >= WARMUP_START) & (raw[dcol] <= Pm_full.index.max())]

            if len(raw) == 0:
                continue

            flag = raw[fcol]
            if str(flag.dtype).lower() == "bool":
                mask = flag.fillna(False)
            else:
                mask = pd.to_numeric(flag, errors="coerce").fillna(0) > 0

            tick_set = set(raw.loc[mask, tcol].astype(str).str.upper().unique().tolist())
            if len(tick_set) > 0:
                return tick_set, path
        except Exception:
            continue

    return set(), None

sp500_set, sp500_source = _load_sp500_set_from_raw()
if len(sp500_set) > 0:
    inter = sorted(set(Pm_full.columns).intersection(sp500_set))
    if len(inter) >= TOP_N:
        Pm_full = Pm_full.reindex(columns=inter)
        px_close = px_close.reindex(columns=inter)
        if vol_daily is not None:
            vol_daily = vol_daily.reindex(columns=inter)
        if open_daily is not None:
            open_daily = open_daily.reindex(columns=inter)
        print(f"Universo intersectado con in_sp500 desde: {sp500_source} -> {len(inter)} tickers")
    else:
        print("Aviso: interseccion in_sp500 deja menos de TOP_N; no se aplica para evitar romper notebook.")

# 6) Sin sesgo futuro: exigimos historial minimo pre-backtest (13 observaciones mensuales)
pre_start_monthly = Pm_full[Pm_full.index < BACKTEST_START]
if pre_start_monthly.shape[0] < 13:
    raise ValueError(
        f"Warm-up insuficiente: solo {pre_start_monthly.shape[0]} meses antes del backtest; se requieren al menos 13."
    )

has_hist_13m = pre_start_monthly.notna().sum(axis=0) >= 13

# 7) Filtro anti-outliers usando datos diarios pre-backtest
pre_start_daily = px_close[(px_close.index < BACKTEST_START) & (px_close.index >= WARMUP_START)].copy()
obs_ok = pre_start_daily.notna().sum(axis=0) >= MIN_DAILY_OBS_PRE

if pre_start_daily.shape[0] == 0:
    raise ValueError("No hay datos diarios en ventana pre-backtest para filtros anti-outliers.")

last_pre_price = pre_start_daily.ffill().iloc[-1]
price_ok = last_pre_price >= MIN_LAST_PRICE_PRE

ret_pre = pre_start_daily.pct_change().replace([np.inf, -np.inf], np.nan)
p99_abs = ret_pre.abs().quantile(0.99)
max_abs = ret_pre.abs().max()
vol_ann = ret_pre.std(skipna=True) * np.sqrt(252)

stable_ret_ok = p99_abs <= MAX_P99_ABS_RET_PRE
max_jump_ok = max_abs <= MAX_MAX_ABS_RET_PRE
vol_ok = vol_ann <= MAX_ANNUAL_VOL_PRE

quality_mask = has_hist_13m & obs_ok & price_ok & stable_ret_ok & max_jump_ok & vol_ok

universe_tickers = sorted(set(quality_mask[quality_mask].index.tolist()))
removed_tickers = sorted(set(Pm_full.columns) - set(universe_tickers))

Pm = Pm_full[universe_tickers].copy()
if vol_daily is not None:
    vol_daily = vol_daily.reindex(columns=Pm.columns)
if open_daily is not None:
    open_daily = open_daily.reindex(columns=Pm.columns)

# Filtros dinamicos anti-gaps (sin look-ahead) para seleccion mensual.
price_lag_monthly = Pm.shift(1)
ret_d_all = px_close.reindex(columns=Pm.columns).pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
jump63_daily = ret_d_all.abs().rolling(window=JUMP63_WINDOW_DAYS, min_periods=JUMP63_MIN_PERIODS).max()
jump63_monthly = jump63_daily.shift(1).reindex(index=Pm.index, method="ffill").reindex(columns=Pm.columns)

if USE_OPEN_CLOSE_ANOMALY_FILTER and (open_daily is not None) and (open_daily.notna().sum().sum() > 0):
    ratio_lo = 1.0 / float(OPEN_CLOSE_RATIO_MAX)
    ratio_hi = float(OPEN_CLOSE_RATIO_MAX)
    oc_ratio_daily = (open_daily.reindex(index=px_close.index, columns=Pm.columns) / px_close.reindex(columns=Pm.columns)).replace([np.inf, -np.inf], np.nan)
    oc_anomaly_daily = ((oc_ratio_daily < ratio_lo) | (oc_ratio_daily > ratio_hi)) & oc_ratio_daily.notna()
    oc_anomaly_count_cum = oc_anomaly_daily.cumsum().astype(float)
    oc_obs_count_cum = oc_ratio_daily.notna().cumsum().astype(float)
    oc_anomaly_count_monthly = oc_anomaly_count_cum.shift(1).reindex(index=Pm.index, method="ffill").reindex(columns=Pm.columns)
    oc_obs_count_monthly = oc_obs_count_cum.shift(1).reindex(index=Pm.index, method="ffill").reindex(columns=Pm.columns)
    open_close_filter_active = True
else:
    oc_anomaly_count_monthly = pd.DataFrame(index=Pm.index, columns=Pm.columns, dtype=float)
    oc_obs_count_monthly = pd.DataFrame(index=Pm.index, columns=Pm.columns, dtype=float)
    open_close_filter_active = False
    if USE_OPEN_CLOSE_ANOMALY_FILTER:
        msg = "No hay OPEN diario valido para filtro OPEN/CLOSE. Requiere fuente raw con columna OPEN y rejecutar Notebook 3 completo."
        if REQUIRE_OPEN_FOR_OC_FILTER:
            raise ValueError(msg)
        print("Aviso OPEN/CLOSE:", msg)

if USE_ADV20_FILTER and (vol_daily is not None) and (vol_daily.notna().sum().sum() > 0):
    dollar_vol_daily = (px_close.reindex(columns=Pm.columns) * vol_daily.reindex(index=px_close.index, columns=Pm.columns))
    adv20_daily = dollar_vol_daily.rolling(window=ADV20_WINDOW_DAYS, min_periods=ADV20_MIN_PERIODS).mean()
    adv20_monthly = adv20_daily.shift(1).reindex(index=Pm.index, method="ffill").reindex(columns=Pm.columns)
    adv20_filter_active = True
else:
    adv20_monthly = pd.DataFrame(index=Pm.index, columns=Pm.columns, dtype=float)
    adv20_filter_active = False
    if USE_ADV20_FILTER:
        print("Aviso ADV20: no hay volumen diario valido; filtro ADV20 queda inactivo en este run.")

# RV63 mensual sin look-ahead para filtrar activos extremadamente volatiles (opcional).
if USE_RV63_FILTER:
    px_for_rv = px_close.reindex(columns=Pm.columns).sort_index().copy()

    has_daily_like = False
    if isinstance(px_for_rv.index, pd.DatetimeIndex) and len(px_for_rv) > 0:
        obs_by_month = px_for_rv.groupby(px_for_rv.index.to_period("M")).size()
        has_daily_like = (obs_by_month.median() > 1)

    if (not has_daily_like) or (px_for_rv.shape[0] < (RV63_MIN_PERIODS + 1)) or (px_for_rv.shape[1] == 0):
        rv63_monthly = pd.DataFrame(index=Pm.index, columns=Pm.columns, dtype=float)
        print("Aviso RV63: no hay panel diario suficiente; filtro RV63 queda inactivo en este run.")
    else:
        rv63_daily = compute_realized_vol_63(px_for_rv, window=RV63_WINDOW_DAYS, min_periods=RV63_MIN_PERIODS)
        rv63_daily_lag = rv63_daily.shift(1)
        rv63_monthly = rv63_daily_lag.reindex(index=Pm.index, method="ffill").reindex(columns=Pm.columns)
else:
    rv63_monthly = pd.DataFrame(index=Pm.index, columns=Pm.columns, dtype=float)
    print("RV63 desactivado (USE_RV63_FILTER=False): seleccion momentum puro.")

if Pm.shape[1] < TOP_N:
    raise ValueError(
        f"Universo insuficiente tras limpieza/outliers: {Pm.shape[1]} activos validos. Se requieren al menos {TOP_N}."
    )

print("Rango mensual (con warm-up):", Pm.index.min(), "->", Pm.index.max())
print("Meses pre-backtest:", pre_start_monthly.shape[0])
print("Numero de activos en universo final limpio (sin GLD):", Pm.shape[1])
print("Filtro precio lag mensual >=", MIN_PRICE_FILTER_MONTHLY_LAG)
print("Filtro jump63 activo:", USE_JUMP63_FILTER, "| max_jump63:", MAX_JUMP63_ABS_RET)
print("Filtro ADV20 activo:", bool(USE_ADV20_FILTER and adv20_filter_active), "| min_adv20:", MIN_ADV20_DOLLAR)
print("Filtro OPEN/CLOSE anomalias activo:", bool(USE_OPEN_CLOSE_ANOMALY_FILTER and open_close_filter_active),
      "| ratio_max:", OPEN_CLOSE_RATIO_MAX, "| max_days:", OPEN_CLOSE_ANOMALY_MAX_DAYS)

if USE_ADV20_FILTER and adv20_filter_active:
    adv20_stack = adv20_monthly.stack().dropna()
    if len(adv20_stack) > 0:
        print("ADV20 mensual (lagged) median/p90:", adv20_stack.quantile(0.5), adv20_stack.quantile(0.9))

if USE_JUMP63_FILTER:
    jump63_stack = jump63_monthly.stack().dropna()
    if len(jump63_stack) > 0:
        print("Jump63 mensual (lagged) quantiles:")
        print(jump63_stack.quantile([0.5, 0.9, 0.95, 0.99]))

if USE_OPEN_CLOSE_ANOMALY_FILTER and open_close_filter_active:
    last_anom = oc_anomaly_count_monthly.iloc[-1].dropna()
    if len(last_anom) > 0:
        print("OPEN/CLOSE anomalias acumuladas (ultimo rebalance) quantiles:")
        print(last_anom.quantile([0.5, 0.9, 0.95, 0.99]))

if USE_RV63_FILTER:
    rv63_stack = rv63_monthly.stack().dropna()
    if len(rv63_stack) > 0:
        print("RV63 mensual (lagged) quantiles:")
        print(rv63_stack.quantile([0.5, 0.9, 0.95, 0.99]))
    else:
        print("RV63 mensual (lagged): sin datos suficientes en este punto.")
else:
    print("RV63: desactivado en esta corrida.")

print("Activos eliminados por filtros anti-outliers:", len(removed_tickers))
if len(removed_tickers) > 0:
    print("Ejemplo eliminados:", removed_tickers[:25])


OPEN diario cargado desde source_path para filtro de anomalias OPEN/CLOSE.


C:\Users\alons\AppData\Local\Temp\ipykernel_25656\2657121039.py:141: FutureWarning: 'BM' is deprecated and will be removed in a future version, please use 'BME' instead.
  Pm = px.resample("BM").last()


Universo intersectado con in_sp500 desde: C:\\Users\\alons\\Desktop\\Práctica 7\\sp500_history.parquet -> 603 tickers
RV63 desactivado (USE_RV63_FILTER=False): seleccion momentum puro.
Rango mensual (con warm-up): 2013-12-31 00:00:00 -> 2026-01-30 00:00:00
Meses pre-backtest: 13
Numero de activos en universo final limpio (sin GLD): 533
Filtro precio lag mensual >= 5.0
Filtro jump63 activo: True | max_jump63: 0.7
Filtro ADV20 activo: False | min_adv20: 2000000.0
Filtro OPEN/CLOSE anomalias activo: True | ratio_max: 5.0 | max_days: 10
Jump63 mensual (lagged) quantiles:
0.50    0.053894
0.90    0.123005
0.95    0.158485
0.99    0.263406
dtype: float64
OPEN/CLOSE anomalias acumuladas (ultimo rebalance) quantiles:
0.50    0.0
0.90    0.0
0.95    0.0
0.99    0.0
Name: 2026-01-30 00:00:00, dtype: float64
RV63: desactivado en esta corrida.
Activos eliminados por filtros anti-outliers: 70
Ejemplo eliminados: ['ABNB', 'AIV', 'AMCR', 'AMD', 'AMTM', 'ANET', 'APP', 'ARES', 'BHF', 'CARR', 'CEG', 'CF

C:\Users\alons\AppData\Local\Temp\ipykernel_25656\2470170939.py:185: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret_pre = pre_start_daily.pct_change().replace([np.inf, -np.inf], np.nan)


## Paso A/B/C/D/E/F/G: Momentum, Z-score y filtros anti-gaps (sin GLD)

Paso A:
- `R12 = log(Pm.shift(1)/Pm.shift(13))`
- `R6  = log(Pm.shift(1)/Pm.shift(7))`

Paso B:
- Z-score cross-sectional por mes (`Z12`, `Z6`).

Paso C:
- `score = 0.5*(Z12 + Z6)`

Paso D (anti-gaps):
- Precio minimo mensual lagged (`price_lag >= MIN_PRICE_FILTER_MONTHLY_LAG`).

Paso E (anti-gaps):
- Liquidez minima `ADV20` en dolares (lagged, sin look-ahead) cuando hay volumen disponible.

Paso F (anti-gaps):
- Exclusion por salto extremo reciente: max `|ret_d|` de 63 dias (lagged) <= umbral.

Paso G (anti-gaps OPEN/CLOSE):
- Exclusion por historial anomalo de `OPEN/CLOSE` (sin look-ahead, acumulado y lagged).

Seleccion final:
- Ranking descendente por score y TOP 20 por fecha de rebalanceo.
- Si en un mes hay <20 candidatos validos: mantener pesos del mes anterior (hold-last-weights).

Nota:
- RV63 queda desactivado en este modo (`USE_RV63_FILTER=False`).


In [41]:
top_n = TOP_N

R12, R6 = compute_momentum(Pm)

# Primera fecha valida basada en SCORES validos (>= top_n) y >= inicio backtest
Z12_tmp = cross_sectional_zscore(R12)
Z6_tmp = cross_sectional_zscore(R6)
score_tmp = 0.5 * (Z12_tmp + Z6_tmp)

valid_score_count = score_tmp.notna().sum(axis=1)
first_rebalance = valid_score_count[(valid_score_count >= top_n) & (valid_score_count.index >= BACKTEST_START)].index.min()

if pd.isna(first_rebalance):
    raise ValueError(
        f"No existe ninguna fecha >= {BACKTEST_START.date()} con al menos {top_n} scores validos."
    )

print("Primera fecha de rebalanceo ejecutable (SCORE>=20 y >= inicio):", first_rebalance)
print("Resumen de activos con score valido por fecha:")
print(valid_score_count.describe())
print("RV63 filter activo:", USE_RV63_FILTER)
print("RV63 ventana/min_periods:", RV63_WINDOW_DAYS, RV63_MIN_PERIODS)
print("RV63 max percentile:", RV63_MAX_PERCENTILE)
print("Price lag min:", MIN_PRICE_FILTER_MONTHLY_LAG)
print("ADV20 filter activo:", bool(USE_ADV20_FILTER and adv20_filter_active), "| min_adv20:", MIN_ADV20_DOLLAR)
print("Jump63 filter activo:", USE_JUMP63_FILTER, "| max_jump63:", MAX_JUMP63_ABS_RET)
print("OPEN/CLOSE anomaly filter activo:", bool(USE_OPEN_CLOSE_ANOMALY_FILTER and open_close_filter_active),
      "| ratio_max:", OPEN_CLOSE_RATIO_MAX, "| max_days:", OPEN_CLOSE_ANOMALY_MAX_DAYS, "| min_obs:", OPEN_CLOSE_MIN_OBS)

selection_wide, selection_long = build_scores_and_select(
    R12,
    R6,
    top_n=top_n,
    hold_last_weights=True,
    rv63=rv63_monthly,
    use_rv63_filter=USE_RV63_FILTER,
    rv63_max_percentile=RV63_MAX_PERCENTILE,
    price_lag=price_lag_monthly,
    min_price_lag=MIN_PRICE_FILTER_MONTHLY_LAG,
    adv20=adv20_monthly,
    use_adv20_filter=(USE_ADV20_FILTER and adv20_filter_active),
    min_adv20=MIN_ADV20_DOLLAR,
    jump63=jump63_monthly,
    use_jump63_filter=USE_JUMP63_FILTER,
    max_jump63_abs_ret=MAX_JUMP63_ABS_RET,
    oc_anomaly_count=oc_anomaly_count_monthly,
    oc_obs_count=oc_obs_count_monthly,
    use_open_close_anomaly_filter=(USE_OPEN_CLOSE_ANOMALY_FILTER and open_close_filter_active),
    open_close_anomaly_max_days=OPEN_CLOSE_ANOMALY_MAX_DAYS,
    open_close_min_obs=OPEN_CLOSE_MIN_OBS,
)

# Export/ejecucion solo desde inicio de backtest (warm-up solo para calculo de senales)
selection_wide = selection_wide[selection_wide.index >= BACKTEST_START].copy()
selection_long = selection_long[selection_long["rebalance_date"] >= BACKTEST_START].copy()

# Garantiza maximo TOP_N por fecha en salida
selection_long = (
    selection_long.sort_values(["rebalance_date", "rank", "ticker"], ascending=[True, True, True])
    .groupby("rebalance_date", as_index=False, group_keys=False)
    .head(top_n)
    .reset_index(drop=True)
)

print("\nSanity checks Pm:")
print("Pm shape:", Pm.shape)
print("Pm rango:", Pm.index.min(), "->", Pm.index.max())
print("Pm infer_freq:", pd.infer_freq(Pm.index))
print("Ejemplo fechas Pm:", list(Pm.index[:6]))

print("\nRango final exportado (selection_long):",
      selection_long["rebalance_date"].min(), "->", selection_long["rebalance_date"].max())
print("Fechas rebalance exportadas:", selection_long["rebalance_date"].nunique())

print("\nPreview selection_long:")
display(selection_long.head(30))

print("Shape selection_long:", selection_long.shape)
print("Shape selection_wide:", selection_wide.shape)


Primera fecha de rebalanceo ejecutable (SCORE>=20 y >= inicio): 2015-01-30 00:00:00
Resumen de activos con score valido por fecha:
count    146.000000
mean     485.541096
std      152.322576
min        0.000000
25%      533.000000
50%      533.000000
75%      533.000000
max      533.000000
dtype: float64
RV63 filter activo: False
RV63 ventana/min_periods: 63 40
RV63 max percentile: 0.9
Price lag min: 5.0
ADV20 filter activo: False | min_adv20: 2000000.0
Jump63 filter activo: True | max_jump63: 0.7
OPEN/CLOSE anomaly filter activo: True | ratio_max: 5.0 | max_days: 10 | min_obs: 60

Sanity checks Pm:
Pm shape: (146, 533)
Pm rango: 2013-12-31 00:00:00 -> 2026-01-30 00:00:00
Pm infer_freq: BME
Ejemplo fechas Pm: [Timestamp('2013-12-31 00:00:00'), Timestamp('2014-01-31 00:00:00'), Timestamp('2014-02-28 00:00:00'), Timestamp('2014-03-31 00:00:00'), Timestamp('2014-04-30 00:00:00'), Timestamp('2014-05-30 00:00:00')]

Rango final exportado (selection_long): 2015-01-30 00:00:00 -> 2026-01-30 0

,rebalance_date,ticker,rank,score,z12,z6,r12,r6,rv63,rv63_cutoff,rv63_filter_pass,price_lag,price_filter_pass,adv20,adv20_filter_pass,jump63,jump63_filter_pass,oc_anomaly_count,oc_obs_count,oc_anomaly_filter_pass,weight,rebalanced
0,2015-01-30,SWKS,1,3.045842,3.866911,2.224773,0.941386,0.441439,NaN,NaN,NaN,58.489437,1,NaN,NaN,0.046198,1,0.0,292.0,1,0.05,1
1,2015-01-30,ENPH,2,2.946168,3.244474,2.647861,0.812681,0.513629,NaN,NaN,NaN,14.290000,1,NaN,NaN,0.227564,1,0.0,292.0,1,0.05,1
2,2015-01-30,LUV,3,2.792901,3.263995,2.321806,0.816718,0.457995,NaN,NaN,NaN,37.373207,1,NaN,NaN,0.084150,1,0.0,292.0,1,0.05,1
3,2015-01-30,AXON,4,2.730287,1.787089,3.673484,0.511329,0.688626,NaN,NaN,NaN,26.480000,1,NaN,NaN,0.086739,1,0.0,292.0,1,0.05,1
4,2015-01-30,PANW,5,2.419938,2.977219,1.862656,0.757419,0.379653,NaN,NaN,NaN,20.428312,1,NaN,NaN,0.061187,1,0.0,292.0,1,0.05,1
5,2015-01-30,UAL,6,2.283446,2.070574,2.496318,0.569947,0.487772,NaN,NaN,NaN,66.889999,1,NaN,NaN,0.081802,1,0.0,292.0,1,0.05,1
6,2015-01-30,EW,7,2.231246,2.511708,1.950783,0.661163,0.394690,NaN,NaN,NaN,21.229977,1,NaN,NaN,0.034279,1,0.0,292.0,1,0.05,1
7,2015-01-30,AVGO,8,2.066882,2.499615,1.634148,0.658662,0.340664,NaN,NaN,NaN,7.705092,1,NaN,NaN,0.083465,1,0.0,292.0,1,0.05,1
8,2015-01-30,RCL,9,2.033461,2.074087,1.992836,0.570673,0.401865,NaN,NaN,NaN,72.225250,1,NaN,NaN,0.066180,1,0.0,292.0,1,0.05,1
9,2015-01-30,EA,10,2.003944,2.784572,1.223317,0.717585,0.270565,NaN,NaN,NaN,45.712391,1,NaN,NaN,0.128073,1,0.0,292.0,1,0.05,1


Shape selection_long: (2660, 22)
Shape selection_wide: (133, 533)


## Export CSV obligatorio

Se guarda `selection_long` en:
- `outputs/selected_top20_by_rebalance.csv`

Si falla, fallback a:
- `./selected_top20_by_rebalance.csv`


In [42]:
save_primary = "outputs/selected_top20_by_rebalance.csv"
save_fallback = "selected_top20_by_rebalance.csv"
saved_path = None

try:
    fs.LocalFileSystem().create_dir("outputs", recursive=True)
    selection_long.to_csv(save_primary, index=False)
    saved_path = save_primary
except Exception as e1:
    print("No se pudo guardar en outputs/:", str(e1))

if saved_path is None:
    try:
        selection_long.to_csv(save_fallback, index=False)
        saved_path = save_fallback
    except Exception as e2:
        raise RuntimeError(f"No se pudo guardar el CSV ni en outputs/ ni en raiz. Detalle: {e2}")

print("CSV guardado en:", saved_path)
print("Filas guardadas:", len(selection_long))
if len(selection_long) > 0:
    print("Rango de fechas guardado:", selection_long["rebalance_date"].min(), "->", selection_long["rebalance_date"].max())


CSV guardado en: outputs/selected_top20_by_rebalance.csv
Filas guardadas: 2660
Rango de fechas guardado: 2015-01-30 00:00:00 -> 2026-01-30 00:00:00


## Resultado final para Notebook 4

Variables disponibles:
- `Pm`: precios mensuales del universo final.
- `R12`, `R6`: momentum logaritmico con lag de 1 mes.
- `selection_wide`: tickers seleccionados por rank para cada rebalanceo.
- `selection_long`: trazabilidad completa de score y factores.

En este notebook NO se implementan costes ni ejecucion de ordenes (eso corresponde a Notebook 4).
